# F3_eval (Colab GPU): answer and judge a prompt set with one model (base or a merged adapter)
Copied from `bilingual/notebooks/B2_eval.ipynb`. `PSET`: `half` (pool-half US + Japan) or `e4` (E4 matched-cue prompts). `SRC`: `base`, an existing adapter uploaded to `inputs/adapters/<name>`, or a follow-up adapter in `out/adapters/<name>`.

In [1]:
SRC = "bi_h2_s0"
PSET = "half"
WORK = "/content/fu"
MERGED_ROOT = "/content/merged"
ANSWER_BATCH = 400
JUDGE_BATCH = 800


In [2]:
# Parameters
SRC = "bi_h2_s2"
PSET = "half"


In [3]:
# HF_TOKEN: taken from the environment, else from the HF token file on the VM. Never printed or saved.
import os
from pathlib import Path as _P
if not os.environ.get("HF_TOKEN"):
    _tf = _P.home() / ".cache" / "huggingface" / "token"
    if _tf.exists():
        os.environ["HF_TOKEN"] = _tf.read_text().strip()
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not available"


In [4]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


# FU deviation 4 (G4 unavailable, run on L4 24 GB): the fractions in these notebooks were sized for the 96 GB G4.
# On a GPU of 80 GB or less they are scaled by 0.88/0.47, so the largest pair loaded together (0.35 + 0.12) fills <= 88%.
_load_llm_g4 = load_llm


def load_llm(model_id, dtype, gpu_memory_utilization, *a, **kw):
    import torch
    tot = torch.cuda.get_device_properties(0).total_memory / 2**30
    u = gpu_memory_utilization if tot > 80 else round(min(0.88, gpu_memory_utilization * 0.88 / 0.47), 3)
    if u != gpu_memory_utilization:
        print(f"load_llm: {model_id} gpu_memory_utilization {gpu_memory_utilization} -> {u} ({tot:.0f} GiB GPU)")
    return _load_llm_g4(model_id, dtype, u, *a, **kw)


In [5]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


GPU: NVIDIA L4 (24 GB) | hardware NVIDIA L4 | dtype bfloat16
{'torch': '2.14.0+cu130', 'vllm': '0.33.0', 'transformers': '5.20.2', 'peft': '0.21.2', 'python': '3.13.16'}


In [6]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [7]:
# free_llm, copied verbatim from 04_eval_batch (cell 5) / J1_chunk_en (cell 9).
def free_llm(llm):
    eng = getattr(llm, "llm_engine", None)
    core = getattr(eng, "engine_core", None)
    if core is not None and hasattr(core, "shutdown"):
        core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

In [8]:
# Inputs, prompt set, adapter. Resumes from STAGE checkpoints; skips if the output folder's DONE checks out.
IN = Path(WORK) / "inputs"
assert (IN / "plan.DONE").read_text().strip() == sha256_file(IN / "plan.json"), "plan.DONE mismatch"
plan = json.loads((IN / "plan.json").read_text())
if PSET == "half":
    assert sha256_file(IN / "eval_half.jsonl") == plan["hashes"]["eval_half.jsonl"]
    PROMPTS = load_jsonl(IN / "eval_half.jsonl")
    assert len(PROMPTS) == plan["n_eval_half"]
elif PSET == "e4":
    e4d = IN / "e4"
    assert (e4d / "DONE").read_text().strip() == sha256_file(e4d / "manifest.json")
    e4man = json.loads((e4d / "manifest.json").read_text())
    assert sha256_file(e4d / "prompts_matched.jsonl") == e4man["files"]["prompts_matched.jsonl"]["sha256"]
    PROMPTS = load_jsonl(e4d / "prompts_matched.jsonl")
else:
    raise ValueError(PSET)
MODEL = f"{PSET}__{SRC}"
OUTD = Path(WORK) / "out" / "eval" / MODEL
STAGE = Path(WORK) / "work" / MODEL
OUTD.mkdir(parents=True, exist_ok=True)
STAGE.mkdir(parents=True, exist_ok=True)


def dir_done(d):
    if not (d / "DONE").exists() or (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


SKIP = dir_done(OUTD)
if SRC == "base":
    ADIR, adapter_sha, adapter_source = None, None, None
else:
    own = Path(WORK) / "out" / "adapters" / SRC
    if dir_done(own):
        ADIR, adapter_source = own, f"followups/adapters/{SRC}"
    else:
        ADIR, adapter_source = IN / "adapters" / SRC, f"uploaded existing adapter {SRC}"
    assert (ADIR / "adapter_model.safetensors").exists(), f"adapter {SRC} missing"
    adapter_sha = {f: sha256_file(ADIR / f) for f in ("adapter_config.json", "adapter_model.safetensors")}
    exp = plan.get("adapter_sha256", {}).get(SRC)
    if exp:
        assert all(adapter_sha[f] == exp[f] for f in adapter_sha), "adapter differs from the plan's recorded hash"
SETS = sorted({p["culture"] for p in PROMPTS})
print(f"MODEL {MODEL}: source {SRC}, cultures {SETS}, {len(PROMPTS)} prompts; skip={SKIP}; {HARDWARE} {DTYPE}")


MODEL half__bi_h2_s2: source bi_h2_s2, cultures ['japan', 'us'], 6598 prompts; skip=False; NVIDIA L4 bfloat16


In [9]:
# Merge the adapter into bf16 weights on the VM's local disk (merge code from 04_eval_batch / J1_chunk_en), then answer in batches of 400.
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
ANS = STAGE / "answers.jsonl"
ans = {r["prompt_id"]: r for r in load_jsonl(ANS)}
todo = [p for p in PROMPTS if p["prompt_id"] not in ans]
print(f"answers: {len(ans)} done, {len(todo)} to go")
if not SKIP and todo:
    if SRC == "base":
        MPATH = TINY_AYA
    else:
        mp = Path(MERGED_ROOT) / SRC
        if not (mp / "config.json").exists():
            b = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
            PeftModel.from_pretrained(b, str(ADIR)).merge_and_unload().save_pretrained(mp)
            AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(mp)
            del b
            gc.collect()
        MPATH = str(mp)
    llm = load_llm(MPATH, DTYPE, gpu_memory_utilization=0.40)
    t0 = time.time()
    for i in range(0, len(todo), ANSWER_BATCH):
        ch = todo[i:i + ANSWER_BATCH]
        outs = answer(llm, [p["prompt"] for p in ch])
        rs = [{"model": MODEL, "prompt_id": p["prompt_id"], **o} for p, o in zip(ch, outs)]
        append_jsonl(ANS, rs)
        ans.update({r["prompt_id"]: r for r in rs})
        print(f"  {MODEL}: {i + len(ch)}/{len(todo)} ({time.time() - t0:.0f}s)")
    free_llm(llm)
assert SKIP or all(p["prompt_id"] in ans for p in PROMPTS)

answers: 0 done, 6598 to go


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

load_llm: /content/merged/bi_h2_s2 gpu_memory_utilization 0.4 -> 0.749 (22 GiB GPU)


W1115 14:52:39.779000 24312 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead


W1115 14:52:39.780000 24312 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead


... (27 outputs left out)

(MainProcess pid=24312) INFO 11-15 15:04:19 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


(MainProcess pid=24312) INFO 11-15 15:04:24 [core_client.py:796] [shutdown] MPClient: engine manager stopped


(MainProcess pid=24312) INFO 11-15 15:04:24 [core_client.py:797] [shutdown] MPClient: cleaning up background resources


(MainProcess pid=24312) INFO 11-15 15:04:24 [core_client.py:799] [shutdown] MPClient: complete


In [10]:
# Judge every answer with both guards, one loaded at a time (loop from 04_eval_batch / J1_chunk_en).
by_pid = {p["prompt_id"]: p for p in PROMPTS}
verdicts = {}
if not SKIP:
    for tag, gid, util in [("g06", GUARD_06, 0.15), ("g4b", GUARD_4B, 0.30)]:
        path = STAGE / f"judge_{tag}.jsonl"
        v = {r["prompt_id"]: r for r in load_jsonl(path)}
        todo_j = [p["prompt_id"] for p in PROMPTS if p["prompt_id"] not in v]
        print(f"{tag}: {len(v)} judged, {len(todo_j)} to go")
        if todo_j:
            g = load_llm(gid, DTYPE, gpu_memory_utilization=util, max_model_len=8192)
            for i in range(0, len(todo_j), JUDGE_BATCH):
                ch = todo_j[i:i + JUDGE_BATCH]
                js = judge(g, [by_pid[k]["prompt"] for k in ch], [ans[k]["answer"] for k in ch])
                rs = [{"prompt_id": k, **j} for k, j in zip(ch, js)]
                append_jsonl(path, rs)
                v.update({r["prompt_id"]: r for r in rs})
                print(f"  {tag}: {i + len(ch)}/{len(todo_j)}")
            free_llm(g)
        verdicts[tag] = v

g06: 0 judged, 6598 to go
load_llm: Qwen/Qwen3Guard-Gen-0.6B gpu_memory_utilization 0.15 -> 0.281 (22 GiB GPU)
(MainProcess pid=24312) INFO 11-15 15:04:25 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.281, 'disable_log_stats': True, 'model': 'Qwen/Qwen3Guard-Gen-0.6B'}


(MainProcess pid=24312) INFO 11-15 15:04:26 [model.py:744] Resolved architecture: Qwen3ForCausalLM


(MainProcess pid=24312) INFO 11-15 15:04:26 [model.py:2170] Using max model len 8192


(MainProcess pid=24312) INFO 11-15 15:04:27 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (23 outputs left out)

  g4b: 5600/6598


  g4b: 6400/6598


  g4b: 6598/6598
(MainProcess pid=24312) INFO 11-15 15:13:59 [core_client.py:792] [shutdown] MPClient: start timeout=default


(MainProcess pid=24312) INFO 11-15 15:13:59 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


In [11]:
# Write eval.jsonl, then manifest.json, then DONE.
def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


if not SKIP:
    META = ["culture", "us_prompt_id", "seed_id", "kind", "framing", "lang", "severity", "category", "expected", "subtopic", "form", "batch"]
    out_rows = []
    for p in PROMPTS:
        a = ans[p["prompt_id"]]
        row = {"model": MODEL, "prompt_id": p["prompt_id"], **{m: p[m] for m in META}, "answer": a["answer"], "tokens": a["tokens"],
               "finish_reason": a["finish_reason"], "answer_lang": answer_lang(a["answer"]), "source": "this_session"}
        for tag in ("g06", "g4b"):
            j = verdicts[tag][p["prompt_id"]]
            for f in ("label", "categories", "unsafe", "refused", "raw"):
                row[f"{tag}_{f}"] = j[f]
        out_rows.append(row)
    assert len(out_rows) == len(PROMPTS) and len({r["prompt_id"] for r in out_rows}) == len(out_rows)
    assert all(r["g06_label"] is not None and r["g4b_label"] is not None for r in out_rows)
    write_jsonl(OUTD / "eval.jsonl", out_rows)
    from collections import Counter
    man = {"model": MODEL, "source_model": SRC, "sets": list(SETS), "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "hardware": HARDWARE, "gpu_name": GPU_NAME, "dtype": DTYPE, "versions": VERSIONS, "serving": "base" if SRC == "base" else "merged",
           "models": {"base": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)}, "g06": {"id": GUARD_06, "sha": model_sha(GUARD_06)},
                      "g4b": {"id": GUARD_4B, "sha": model_sha(GUARD_4B)}},
           "adapter_source": adapter_source, "adapter_sha256": adapter_sha, "plan_sha256": sha256_file(IN / "plan.json"), "pset": PSET,
           "decoding": {"greedy": True, "max_new_tokens": ANSWER_MAX_NEW_TOKENS, "max_model_len": ANSWER_MAX_MODEL_LEN, "answer_batch": ANSWER_BATCH},
           "rows": dict(Counter(r["culture"] for r in out_rows)),
           "n_unparsed_g06": sum(r["g06_label"] == "Unparsed" for r in out_rows), "n_unparsed_g4b": sum(r["g4b_label"] == "Unparsed" for r in out_rows),
           "n_truncated": sum(r["finish_reason"] == "length" for r in out_rows),
           "files": {"eval.jsonl": {"rows": len(out_rows), "sha256": sha256_file(OUTD / "eval.jsonl")}}}
    (OUTD / "manifest.json").write_text(json.dumps(man, indent=1, ensure_ascii=False))
    (OUTD / "DONE").write_text(sha256_file(OUTD / "manifest.json") + "\n")
    print(json.dumps({k: man[k] for k in ("rows", "n_unparsed_g06", "n_unparsed_g4b", "n_truncated")}))
assert dir_done(OUTD)
print("published", OUTD)

{"rows": {"us": 3312, "japan": 3286}, "n_unparsed_g06": 0, "n_unparsed_g4b": 0, "n_truncated": 15}
published /content/fu/out/eval/half__bi_h2_s2
